In [1]:
import psycopg2, getpass

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    dbname='eudr',
    user='aparnadixit',      # Homebrew PostgreSQL uses your Mac username by default
    password=getpass.getpass('PostgreSQL password (just press Enter if you have none): ')
)
print('Connected to', conn.get_dsn_parameters()['dbname'])

Connected to eudr


In [2]:
create_table = """
DROP TABLE IF EXISTS raw.fact_trade_flows;
CREATE TABLE raw.fact_trade_flows (
    flow_id                            BIGINT PRIMARY KEY,
    source_dataset                     TEXT,
    commodity                          TEXT,
    product_type                       TEXT,
    year                               INTEGER,
    origin_country                     TEXT,
    origin_country_iso2                TEXT,
    origin_country_iso3                TEXT,
    origin_level1_type                 TEXT,
    origin_level1                      TEXT,
    origin_level1_id                   TEXT,
    origin_level2_type                 TEXT,
    origin_level2                      TEXT,
    origin_level2_id                   TEXT,
    biome                              TEXT,
    export_location                    TEXT,
    exporter                           TEXT,
    exporter_group                     TEXT,
    importer                           TEXT,
    importer_group                     TEXT,
    zero_deforestation_commitment      TEXT,
    destination_country                TEXT,
    destination_iso2                   TEXT,
    destination_iso3                   TEXT,
    destination_basis                  TEXT,
    volume_t                           DOUBLE PRECISION,
    trade_value_usd                    DOUBLE PRECISION,
    trade_value_type                   TEXT,
    unit_value_usd_per_t               DOUBLE PRECISION,
    land_use_ha                        DOUBLE PRECISION,
    deforestation_exposure_ha          DOUBLE PRECISION,
    deforestation_exposure_window_yrs  NUMERIC,
    deforestation_annual_ha            DOUBLE PRECISION,
    eudr_relevant_deforestation_ha     DOUBLE PRECISION,
    origin_traceability                TEXT,
    is_origin_known                    BOOLEAN,
    has_deforestation_data             BOOLEAN,
    is_deforestation_linked            BOOLEAN,
    flag_value_outlier                 BOOLEAN,
    flag_value_missing                 BOOLEAN,
    flag_name_encoding_fixed           BOOLEAN
);
"""

with conn, conn.cursor() as cur:
    cur.execute(create_table)
    with open('../clean/fact_trade_flows.csv', 'r', encoding='utf-8') as f:
        cur.copy_expert(
            "COPY raw.fact_trade_flows FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')", f)
    cur.execute('SELECT COUNT(*) FROM raw.fact_trade_flows')
    print('Rows loaded:', f'{cur.fetchone()[0]:,}')

Rows loaded: 336,492


In [3]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT commodity, year, ROUND(SUM(volume_t)) AS volume_t
        FROM raw.fact_trade_flows
        GROUP BY commodity, year
        ORDER BY commodity, year
    """)
    for row in cur.fetchall():
        print(row)
conn.close()

('Beef', 2020, 118768.0)
('Beef', 2021, 119254.0)
('Beef', 2022, 132108.0)
('Beef', 2023, 116586.0)
('Cocoa', 2020, 1345344.0)
('Cocoa', 2021, 1244485.0)
('Cocoa', 2022, 1298093.0)
('Cocoa', 2023, 1083476.0)
('Cocoa', 2024, 1016314.0)
('Coffee', 2020, 385768.0)
('Coffee', 2021, 268401.0)
('Coffee', 2022, 120993.0)
('Palm oil', 2020, 3413281.0)
('Palm oil', 2021, 2970744.0)
('Palm oil', 2022, 2289163.0)
('Palm oil', 2023, 1997092.0)
('Palm oil', 2024, 1895664.0)
('Soy', 2020, 13923690.0)
('Soy', 2021, 14525500.0)
('Soy', 2022, 10304993.0)
('Wood pulp', 2022, 3200.0)
('Wood pulp', 2023, 4.0)
('Wood pulp', 2024, 7743.0)
